In [1]:
from sklearn.linear_model import LogisticRegression
from pathlib import Path
from sklearn.model_selection import StratifiedKFold,train_test_split,GridSearchCV,RandomizedSearchCV,cross_val_predict,LeaveOneGroupOut,cross_validate
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer,MissingIndicator
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.metrics import roc_auc_score,brier_score_loss,average_precision_score,log_loss
import os
from xgboost import XGBClassifier
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [2]:
SEED=42
cv=StratifiedKFold(5,shuffle=True,random_state=SEED)
oof_cv=StratifiedKFold(5,shuffle=True,random_state=SEED+1)
COLS = ['age','sex','cp','trestbps','chol','fbs','restecg','thalach',
        'exang','oldpeak','slope','ca','thal','num']
FILES = {'cleveland':'processed.cleveland.data','hungary':'processed.hungarian.data',
         'switzerland':'processed.switzerland.data','va':'processed.va.data'}
RAW=os.path.join(Path.cwd(),'data/raw/')



In [3]:
frames=[]

for k,v in FILES.items():
    data=pd.read_csv(os.path.join(RAW,'heart+disease',v),names=COLS,na_values='?')
    data['site']=k
    frames.append(data)
df=pd.concat(frames,ignore_index=True)
df.shape

(920, 15)

In [4]:
for c in ['chol','trestbps']:
    df.loc[df[c]==0,c]=np.nan
df['target']=(df['num']>0).astype(int)
df=df.drop(columns='num').drop_duplicates(keep='first')

In [5]:
ROLES = {'cont':['age','trestbps','chol','thalach','oldpeak'],'binary':['sex','fbs'],
         'nominal':['cp','restecg','thal','exang'],'ordinal':['slope','ca']}
high_miss=[v for v in sum(ROLES.values(),[]) if df[v].isna().mean()>0.3]
ROLES= {k: [c for c in v if c not in high_miss] for k,v in ROLES.items()}
print(high_miss)
ROLES

['thal', 'slope', 'ca']


{'cont': ['age', 'trestbps', 'chol', 'thalach', 'oldpeak'],
 'binary': ['sex', 'fbs'],
 'nominal': ['cp', 'restecg', 'exang'],
 'ordinal': []}

In [6]:
NUMERIC=ROLES['cont']+ROLES['ordinal'];CATEGORICAL=ROLES['nominal'];BINARY=ROLES['binary']
FEATURES=NUMERIC+CATEGORICAL+BINARY
X=df[FEATURES].copy()
y=df['target'].copy()
groups=df['site'].copy()
assert len(X)==len(y)

In [7]:
X_tr,X_te,y_tr,y_te=train_test_split(X,y,test_size=0.25,stratify=y,random_state=SEED)
g_tr=groups.loc[X_tr.index]

In [8]:
#native nan handling
Xn=X.copy()
for c in CATEGORICAL:
    Xn[c]=Xn[c].astype('Int64').astype('category')
for c in BINARY:
    Xn[c]=Xn[c].astype('Int64').astype('category')

Xn_tr,Xn_te=Xn.loc[X_tr.index],Xn.loc[X_te.index]
Xn.dtypes,Xn_tr.isna().sum()

(age          float64
 trestbps     float64
 chol         float64
 thalach      float64
 oldpeak      float64
 cp          category
 restecg     category
 exang       category
 sex         category
 fbs         category
 dtype: object,
 age           0
 trestbps     48
 chol        149
 thalach      45
 oldpeak      49
 cp            0
 restecg       0
 exang        45
 sex           0
 fbs          63
 dtype: int64)

In [9]:
num_p=Pipeline([('imp',SimpleImputer(strategy='median')),('scl',StandardScaler())])
bin_p=Pipeline([('imp',SimpleImputer(strategy='most_frequent'))])
nom_p=Pipeline([('imp',SimpleImputer(strategy='most_frequent')),
                 ('ohe',OneHotEncoder(drop='first',handle_unknown='infrequent_if_exist',sparse_output=False))])
def make_pre():
    return ColumnTransformer([
        ('num',num_p,NUMERIC),
        ('bin',bin_p,BINARY),
        ('nom',nom_p,CATEGORICAL),
        ('miss',MissingIndicator(features='all'),['trestbps','thalach','oldpeak'])
    ],remainder='drop')

In [10]:
lr_gs=GridSearchCV(Pipeline([('pre',make_pre()),('clf',LogisticRegression(penalty='elasticnet',solver='saga',max_iter=20000,random_state=SEED))]),{'clf__C':np.logspace(-5,2,15),
                    "clf__l1_ratio":[0.0,0.25,0.5,0.75,1.0]},
                    scoring={'log_loss':'neg_log_loss','auc':'roc_auc'},cv=cv,n_jobs=-1,refit='log_loss').fit(X_tr,y_tr)
lr_best=lr_gs.best_estimator_

In [11]:
grid={'clf__n_estimators':[100,200,300,400,500],'clf__max_depth':[2,3,4,5,6],
      'clf__learning_rate':[0.01,0.03,0.05,0.1,0.2],'clf__min_child_weight':[1,3,5,10],
      'clf__subsample':[0.6,0.8,0.9,1.0],'clf__colsample_bytree':[0.6,.8,1.0],
      'clf__gamma':[0,0.5,1.0],'clf__reg_lambda':[0.1,1.0,5.0,20.0]}

xgbA=RandomizedSearchCV(Pipeline([('pre',make_pre()),('clf',XGBClassifier(objective='binary:logistic',eval_metric='logloss',tree_method='hist',random_state=SEED,n_jobs=-1))]),grid,
                        n_iter=300,random_state=SEED,scoring={'log_loss':'neg_log_loss','auc':'roc_auc'},cv=cv,n_jobs=-1,refit='log_loss').fit(X_tr,y_tr)

ngrid= {k.replace('clf__',''):v for k,v in grid.items()}
xgbB=RandomizedSearchCV(XGBClassifier(objective='binary:logistic',eval_metric='logloss',tree_method='hist',n_jobs=-1,random_state=SEED),ngrid,n_iter=300,random_state=SEED,cv=cv,
                        scoring={'log_loss':'neg_log_loss','auc':'roc_auc'},refit='log_loss',n_jobs=-1).fit(Xn_tr,y_tr)



In [12]:
print('\nLR   best:', {k.replace('clf__',''):(round(v,4) if isinstance(v,float) else v) for k,v in lr_gs.best_params_.items()})
print('XGB-A best:', {k.replace('clf__',''):v for k,v in xgbA.best_params_.items()})
print('XGB-B best:', xgbB.best_params_)


LR   best: {'C': np.float64(0.3162), 'l1_ratio': 0.0}
XGB-A best: {'subsample': 0.9, 'reg_lambda': 20.0, 'n_estimators': 300, 'min_child_weight': 3, 'max_depth': 2, 'learning_rate': 0.03, 'gamma': 1.0, 'colsample_bytree': 1.0}
XGB-B best: {'subsample': 0.9, 'reg_lambda': 20.0, 'n_estimators': 300, 'min_child_weight': 3, 'max_depth': 2, 'learning_rate': 0.03, 'gamma': 1.0, 'colsample_bytree': 1.0}


In [13]:
for name, s in [('LR',lr_gs),('XGB-A',xgbA),('xgb-B',xgbB)]:
    i=s.best_index_
    print(f"{name}: cv logloss={-s.cv_results_['mean_test_log_loss'][i]:.4f}  "
        f"cv auc={s.cv_results_['mean_test_auc'][i]:.4f}")


LR: cv logloss=0.4502  cv auc=0.8678
XGB-A: cv logloss=0.4526  cv auc=0.8690
xgb-B: cv logloss=0.4456  cv auc=0.8723


In [14]:
p_lr  = cross_val_predict(lr_best, X_tr, y_tr, cv=oof_cv, method='predict_proba')[:,1]
p_A   = cross_val_predict(xgbA.best_estimator_, X_tr, y_tr, cv=oof_cv, method='predict_proba')[:,1]
p_B   = cross_val_predict(xgbB.best_estimator_, Xn_tr, y_tr, cv=oof_cv, method='predict_proba')[:,1]

In [15]:
def row(n,yt,p): 
    return {'model':n,'auc':round(roc_auc_score(yt,p),4),
    'ap':round(average_precision_score(yt,p),4),'log_loss':round(log_loss(yt,p),4),
    'brier':round(brier_score_loss(yt,p),4)}

In [16]:
print(pd.DataFrame([row('logreg',y_tr,p_lr),row('xgb pipeline',y_tr,p_A),
                    row('xgb native',y_tr,p_B)]).to_string(index=False))

       model    auc     ap  log_loss  brier
      logreg 0.8694 0.8731    0.4502 0.1438
xgb pipeline 0.8685 0.8771    0.4510 0.1447
  xgb native 0.8715 0.8746    0.4461 0.1417


In [19]:
def paired(yt,p1,p2,B=4000,seed=SEED):
    rng=np.random.default_rng(seed)
    yt=np.asarray(yt)
    pos=np.where(yt==1)[0]
    neg=np.where(yt==0)[0]
    obs=roc_auc_score(yt,p2)-roc_auc_score(yt,p1)
    d=np.empty(B)
    for b in range(B):
        i=np.concatenate([rng.choice(pos,len(pos),True),rng.choice(neg,len(neg),True)])
        d[b]=roc_auc_score(yt[i],p2[i])-roc_auc_score(yt[i],p1[i])
    lo,hi=np.percentile(d,[2.5,97.5])
    return obs,d.std(ddof=1),lo,hi,min(2*min((d<=0).mean(),(d>=0).mean()),1.0)

for nm,p in [('xgb pipeline',p_A),('xgb native',p_B)]:
    o,s,lo,hi,pv = paired(y_tr,p_lr,p)
    print(f'{nm:14s} AUC delta={o:+.4f}  SE={s:.4f}  CI=[{lo:+.4f},{hi:+.4f}]  p={pv:.3f}'
          f'  -> {"SIGNIFICANT" if lo*hi>0 else "within noise"}')
o,s,lo,hi,pv = paired(y_tr,p_A,p_B)
print(f'{"native vs pipe":14s} AUC delta={o:+.4f}  SE={s:.4f}  CI=[{lo:+.4f},{hi:+.4f}]  p={pv:.3f}')


xgb pipeline   AUC delta=-0.0009  SE=0.0045  CI=[-0.0098,+0.0078]  p=0.847  -> within noise
xgb native     AUC delta=+0.0021  SE=0.0051  CI=[-0.0077,+0.0123]  p=0.679  -> within noise
native vs pipe AUC delta=+0.0031  SE=0.0037  CI=[-0.0040,+0.0102]  p=0.393


In [20]:
def paired_loss(yt,p1,p2,metric='log_loss',B=4000,seed=SEED):
    yt=np.asarray(yt); eps=1e-15
    if metric=='log_loss':
        f=lambda p: -(yt*np.log(np.clip(p,eps,1))+(1-yt)*np.log(np.clip(1-p,eps,1)))
    else:
        f=lambda p: (yt-p)**2
    diff=f(p1)-f(p2)                # per-row loss(p1)-loss(p2); >0 means p2 better
    rng=np.random.default_rng(seed)
    pos=np.where(yt==1)[0]; neg=np.where(yt==0)[0]
    d=np.empty(B)
    for b in range(B):
        i=np.concatenate([rng.choice(pos,len(pos),True),rng.choice(neg,len(neg),True)])
        d[b]=diff[i].mean()
    lo,hi=np.percentile(d,[2.5,97.5])
    return diff.mean(),d.std(ddof=1),lo,hi,min(2*min((d<=0).mean(),(d>=0).mean()),1.0)


In [21]:
# test-set predictions (best_estimator_ is already refit on all of X_tr)
q_lr = lr_best.predict_proba(X_te)[:,1]
q_A  = xgbA.best_estimator_.predict_proba(X_te)[:,1]
q_B  = xgbB.best_estimator_.predict_proba(Xn_te)[:,1]

print(pd.DataFrame([row('logreg',y_te,q_lr),row('xgb pipeline',y_te,q_A),
                    row('xgb native',y_te,q_B)]).to_string(index=False))
for nm,q in [('xgb pipeline',q_A),('xgb native',q_B)]:
    o,s,lo,hi,pv=paired(y_te,q_lr,q)
    print(f'{nm:14s} AUC delta={o:+.4f}  SE={s:.4f}  CI=[{lo:+.4f},{hi:+.4f}]  p={pv:.3f}'
          f'  -> {"SIGNIFICANT" if lo*hi>0 else "within noise"}')


       model    auc     ap  log_loss  brier
      logreg 0.8908 0.8908    0.4158 0.1311
xgb pipeline 0.8944 0.8826    0.4126 0.1298
  xgb native 0.9039 0.8997    0.3918 0.1222
xgb pipeline   AUC delta=+0.0035  SE=0.0076  CI=[-0.0120,+0.0178]  p=0.654  -> within noise
xgb native     AUC delta=+0.0131  SE=0.0080  CI=[-0.0025,+0.0293]  p=0.091  -> within noise


In [22]:
for split,yt,(b,a,n) in [('OOF',y_tr,(p_lr,p_A,p_B)),('TEST',y_te,(q_lr,q_A,q_B))]:
    print(f'--- {split}')
    for metric in ['log_loss','brier']:
        for nm,q in [('xgb pipeline',a),('xgb native',n)]:
            o,s,lo,hi,pv=paired_loss(yt,b,q,metric)
            print(f'{nm:14s} {metric:8s} delta={o:+.4f}  SE={s:.4f}  CI=[{lo:+.4f},{hi:+.4f}]  p={pv:.3f}'
                  f'  -> {"SIGNIFICANT" if lo*hi>0 else "within noise"}')


--- OOF
xgb pipeline   log_loss delta=-0.0008  SE=0.0071  CI=[-0.0146,+0.0131]  p=0.917  -> within noise
xgb native     log_loss delta=+0.0041  SE=0.0080  CI=[-0.0113,+0.0201]  p=0.607  -> within noise
xgb pipeline   brier    delta=-0.0009  SE=0.0026  CI=[-0.0060,+0.0042]  p=0.745  -> within noise
xgb native     brier    delta=+0.0021  SE=0.0031  CI=[-0.0039,+0.0081]  p=0.504  -> within noise
--- TEST
xgb pipeline   log_loss delta=+0.0032  SE=0.0111  CI=[-0.0190,+0.0243]  p=0.798  -> within noise
xgb native     log_loss delta=+0.0240  SE=0.0118  CI=[+0.0018,+0.0472]  p=0.037  -> SIGNIFICANT
xgb pipeline   brier    delta=+0.0013  SE=0.0043  CI=[-0.0074,+0.0095]  p=0.792  -> within noise
xgb native     brier    delta=+0.0089  SE=0.0052  CI=[-0.0010,+0.0193]  p=0.076  -> within noise


In [23]:
logo=LeaveOneGroupOut(); out={}
out['logreg']=cross_validate(lr_best,X_tr,y_tr,groups=g_tr,cv=logo,scoring='roc_auc')['test_score']
out['xgb pipeline']=cross_validate(xgbA.best_estimator_,X_tr,y_tr,groups=g_tr,cv=logo,scoring='roc_auc')['test_score']
out['xgb native']=cross_validate(xgbB.best_estimator_,Xn_tr,y_tr,groups=g_tr,cv=logo,scoring='roc_auc')['test_score']
t=pd.DataFrame(out,index=sorted(g_tr.unique())).round(4)
t.loc['MEAN']=t.mean().round(4)
print(t.to_string())

             logreg  xgb pipeline  xgb native
cleveland    0.8532        0.8407      0.8364
hungary      0.8923        0.8595      0.8567
switzerland  0.7319        0.7654      0.7601
va           0.7123        0.7098      0.6953
MEAN         0.7974        0.7939      0.7871


In [24]:
from sklearn.base import clone
Xa_tr, Xa_te = Xn_tr.copy(), Xn_te.copy()
med = Xa_tr['chol'].median()
Xa_tr['chol'] = Xa_tr['chol'].fillna(med); Xa_te['chol'] = Xa_te['chol'].fillna(med)
for c in ['fbs', 'exang']:
    mode = Xa_tr[c].mode()[0]
    Xa_tr[c] = Xa_tr[c].fillna(mode); Xa_te[c] = Xa_te[c].fillna(mode)

xgb_abl = clone(xgbB.best_estimator_).fit(Xa_tr, y_tr)
q_abl = xgb_abl.predict_proba(Xa_te)[:,1]
print(pd.DataFrame([row('logreg',y_te,q_lr), row('xgb native',y_te,q_B),
                    row('xgb native, imputed',y_te,q_abl)]).to_string(index=False))
for metric in ['log_loss','brier']:
    o,s,lo,hi,pv = paired_loss(y_te, q_lr, q_abl, metric)
    print(f'imputed vs LR {metric:8s} delta={o:+.4f} CI=[{lo:+.4f},{hi:+.4f}] p={pv:.3f}'
          f'  -> {"SIGNIFICANT" if lo*hi>0 else "within noise"}')


              model    auc     ap  log_loss  brier
             logreg 0.8908 0.8908    0.4158 0.1311
         xgb native 0.9039 0.8997    0.3918 0.1222
xgb native, imputed 0.8946 0.8894    0.4096 0.1293
imputed vs LR log_loss delta=+0.0062 CI=[-0.0149,+0.0269] p=0.577  -> within noise
imputed vs LR brier    delta=+0.0018 CI=[-0.0068,+0.0105] p=0.693  -> within noise


In [25]:
imp = pd.Series(xgbB.best_estimator_.feature_importances_, index=Xn_tr.columns).sort_values(ascending=False)
print('\nnative XGB gain-weighted importance:'); print(imp.round(4).to_string())


native XGB gain-weighted importance:
cp          0.3195
exang       0.1968
oldpeak     0.0774
age         0.0762
sex         0.0761
thalach     0.0750
chol        0.0750
fbs         0.0532
restecg     0.0271
trestbps    0.0238


## Model selection: why L2 logistic regression was kept over XGBoost

### Question
Does a tree ensemble beat the deployed logistic regression enough to justify the extra complexity?
Two XGBoost variants were compared against LR:
- **XGB pipeline:** the same preprocessing as LR (median/mode imputation, scaling, one-hot encoding, missing indicators for `trestbps`, `thalach`, `oldpeak`).
- **XGB native:** raw features, native categorical handling and native NaN handling.

### Setup
- **Data:** 75/25 stratified train/test split (seed 42). The test set was used only after all tuning was finished.
- **Tuning:**
  - LR: grid over `C` (15 values) × `l1_ratio` (0 to 1), 75 candidates.
  - XGB: 300-candidate random search over depth, learning rate, number of trees, regularisation and subsampling. Both variants used the same candidate list.
  - All three used the same 5-fold stratified CV and were refit on log loss.
- **Evaluation:**
  1. Out-of-fold (OOF) predictions on **new folds** (seed 43), to reduce selection bias from tuning.
  2. Held-out test set.
  3. Leave-one-site-out (LOGO) CV over Cleveland, Hungary, Switzerland and VA.
  4. Ablation: XGB native refit without the missingness signal that LR does not get.
- **Significance:** paired, class-stratified bootstrap (4,000 resamples) of the difference vs LR, for AUC, log loss and Brier score.

### Selected hyperparameters
- **LR:** `C ≈ 0.316`, `l1_ratio = 0`. The elastic-net search picked plain **L2 (ridge)**.
- **XGB (both variants):** `max_depth = 2` (the smallest value in the grid), `reg_lambda = 20`, `gamma = 1`, `learning_rate = 0.03`, 300 trees.
  The search pushed toward the simplest, most heavily regularised trees allowed. That points to little usable non-linear or interaction structure in this data.

### Results

**Out-of-fold, training set (seed-43 folds)**

| model | AUC | AP | log loss | Brier |
|---|---|---|---|---|
| logreg | 0.8694 | 0.8731 | 0.4502 | 0.1438 |
| xgb pipeline | 0.8685 | 0.8771 | 0.4510 | 0.1447 |
| xgb native | 0.8715 | 0.8746 | 0.4461 | 0.1417 |

**Held-out test set (n = 230)**

| model | AUC | AP | log loss | Brier |
|---|---|---|---|---|
| logreg | 0.8908 | 0.8908 | 0.4158 | 0.1311 |
| xgb pipeline | 0.8944 | 0.8826 | 0.4126 | 0.1298 |
| xgb native | 0.9039 | 0.8997 | 0.3918 | 0.1222 |

**Paired bootstrap vs LR** (Δ > 0 means the XGB model is better; 95% CI)

| split | model | metric | Δ | 95% CI | p |
|---|---|---|---|---|---|
| OOF | xgb pipeline | AUC | −0.0009 | [−0.0098, +0.0078] | 0.85 |
| OOF | xgb native | AUC | +0.0021 | [−0.0077, +0.0123] | 0.68 |
| OOF | xgb pipeline | log loss | −0.0008 | [−0.0146, +0.0131] | 0.92 |
| OOF | xgb native | log loss | +0.0041 | [−0.0113, +0.0201] | 0.61 |
| OOF | xgb pipeline | Brier | −0.0009 | [−0.0060, +0.0042] | 0.75 |
| OOF | xgb native | Brier | +0.0021 | [−0.0039, +0.0081] | 0.50 |
| test | xgb pipeline | AUC | +0.0035 | [−0.0120, +0.0178] | 0.65 |
| test | xgb native | AUC | +0.0131 | [−0.0025, +0.0293] | 0.09 |
| test | xgb pipeline | log loss | +0.0032 | [−0.0190, +0.0243] | 0.80 |
| test | xgb native | log loss | **+0.0240** | **[+0.0018, +0.0472]** | **0.037** |
| test | xgb pipeline | Brier | +0.0013 | [−0.0074, +0.0095] | 0.79 |
| test | xgb native | Brier | +0.0089 | [−0.0010, +0.0193] | 0.08 |

**Leave-one-site-out AUC**

| held-out site | logreg | xgb pipeline | xgb native |
|---|---|---|---|
| cleveland | **0.8532** | 0.8407 | 0.8364 |
| hungary | **0.8923** | 0.8595 | 0.8567 |
| switzerland | 0.7319 | **0.7654** | 0.7601 |
| va | **0.7123** | 0.7098 | 0.6953 |
| **mean** | **0.7974** | 0.7939 | 0.7871 |

**Ablation: XGB native with `chol` / `fbs` / `exang` imputed** (train median/mode, no indicator, same hyperparameters; test set)

| model | AUC | log loss | Brier | Δ log loss vs LR [95% CI] | p |
|---|---|---|---|---|---|
| logreg | 0.8908 | 0.4158 | 0.1311 | — | — |
| xgb native (raw NaNs) | 0.9039 | 0.3918 | 0.1222 | +0.0240 [+0.0018, +0.0472] | 0.037 |
| xgb native, imputed | 0.8946 | 0.4096 | 0.1293 | +0.0062 [−0.0149, +0.0269] | 0.58 |

### Interpretation
1. **On OOF, all three models are tied.** Every difference is within noise on AUC, log loss and Brier.
2. **XGB pipeline never beats LR.** With identical inputs, the tree ensemble adds nothing over a linear model.
3. **XGB native has one significant result (test log loss, p = 0.037). On its own it would be weak evidence:**
   - It comes from a single split of 230 rows. The same comparison on OOF (about 690 rows) gives Δ = +0.004, p = 0.61.
   - Brier, the more stable proper score, is not significant on the same test set (p = 0.08).
   - 12 paired tests were run. One p < 0.05 is about what chance alone gives, and it does not survive a Bonferroni correction (α = 0.05/12 ≈ 0.004).
4. **On unseen sites, the ranking reverses.** Under LOGO, XGB native is the **worst** model (mean AUC 0.787 vs 0.797 for LR) and LR is best on 3 of 4 sites.
5. **Mechanism, confirmed by ablation:** XGB native sees raw NaNs in `chol`, `fbs` and `exang`, where LR gets no missing indicator. `chol` was excluded on purpose, because its missingness mostly marks the Switzerland/VA sites, so this missingness acts as a site proxy. These columns also carry non-trivial gain importance in XGB native (`chol` 0.075, `fbs` 0.053).
   Once these three columns are imputed, so XGB native gets the same information as LR, about **75% of its test log loss advantage disappears** (+0.024 → +0.006) and the rest is within noise (p = 0.58). AUC falls from 0.904 to 0.895, about level with LR.
   The significant test result therefore came from exploiting site-linked missingness, not from better modelling of the clinical features. This explains both its random-split gain and its last-place ranking on unseen sites.

### Caveats
- **No nested CV.** Hyperparameters were tuned on the same training data that the OOF predictions come from. XGB picked the best of 300 candidates and LR the best of 75, so this bias favours XGB.
- **XGB native had extra information** (missingness of `chol`/`fbs`/`exang`) that LR deliberately does not get. The ablation shows this accounts for its test-set advantage.
- **The bootstrap CIs treat predictions as fixed** and ignore refit variance, so the true uncertainty is wider.
- **LOGO has only 4 sites,** and the hyperparameters were tuned with every site included.

Every caveat tilts the comparison **toward** XGBoost, and XGBoost still does not show a robust advantage.

### Decision
**Keep L2-regularised logistic regression.**
- No robust performance gain: tied on OOF across all metrics, tied on test AUC and Brier, and XGB native's one significant test win disappears once its site-linked missingness signal is removed.
- Better generalisation to unseen sites, which matters for a multi-site clinical model.
- Interpretable, stable coefficients and odds ratios, already used in the SHAP explanations, the sex/site analysis and the model card.
- Already deployed and monitored. Switching would add complexity and risk for no demonstrated benefit.
- Random forest was not evaluated separately. It is another tree ensemble, and tuned gradient boosting, which usually matches or beats it on small tabular data, showed no robust gain over LR. RF's typically weaker probability calibration would also count against it on log loss, the selection metric.


**Revisit if:** more data becomes available (especially from new sites), or a feature-engineering change gives XGB a gain that holds under both OOF and leave-one-site-out evaluation.
